# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Baseline rule (adapted for a clustering lane):** FlyRank's existing `impression_tier` — a single-dimension, 4-bucket rule (low / moderate / good / excellent), built purely from `impressions_90d`. This is what my clusters need to beat: do they reveal more useful structure than this one dimension already gives a reviewer?

**Reason codes for this baseline:** `low_visibility`, `moderate_visibility`, `good_visibility`, `excellent_visibility` — literally just the tier name, with no reasoning beyond traffic volume.

## 2. Build the ranked queue (writes the CSV)

In [1]:
# Path assumes this notebook lives in work/notebooks/ — adjust if you moved it.
DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"

import pandas as pd
df = pd.read_csv(DATA_PATH)

# Baseline "queue": rank purely by impression_tier, no other signal considered
tier_rank = {"excellent": 0, "good": 1, "moderate": 2, "low": 3}
df["baseline_rank"] = df["impression_tier"].map(tier_rank)
df = df.sort_values("baseline_rank")

import os
os.makedirs("../outputs", exist_ok=True)
df[["content_id","impression_tier","impressions_90d","trend_direction","trend_pct"]]\
    .to_csv("../outputs/baseline_action_score.csv", index=False)
print("Saved ../outputs/baseline_action_score.csv")
print(df["impression_tier"].value_counts())

Saved ../outputs/baseline_action_score.csv
impression_tier
low          11248
moderate     10469
good          7205
excellent     1078
Name: count, dtype: int64


## 3. Top-20 review

In [2]:
# Path assumes this notebook lives in work/notebooks/ — adjust if you moved it.
DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"

import pandas as pd
df = pd.read_csv(DATA_PATH)
tier_rank = {"excellent": 0, "good": 1, "moderate": 2, "low": 3}
df["baseline_rank"] = df["impression_tier"].map(tier_rank)
top20 = df.sort_values("baseline_rank").head(20)
print(top20[["content_id","impression_tier","impressions_90d","trend_direction","engagement_rate"]].to_string(index=False))

          content_id impression_tier  impressions_90d trend_direction  engagement_rate
content_9b934e3e7101       excellent           106384          stable             2.87
content_19960ba60c84       excellent            75748            down             1.93
content_0c3fb4665ee8       excellent            36372          stable             7.84
content_50a9f9f861c6       excellent            41038            down             0.00
content_e203b2c78e5b       excellent            59838            down             1.11
content_88e041922364       excellent            54057            down             2.13
content_aa65fe944a87       excellent            32291              up            10.91
content_f4d42895bc37       excellent            32696            down             1.58
content_8818fd6d967f       excellent            83603            down             7.61
content_7c7b08bd1c9b       excellent            32119          stable             0.99
content_54e9fbc9f066       excellent       

Looking at the top 20 "excellent"-tier pages: several have a declining `trend_direction` and low `engagement_rate` despite their high tier — exactly the blind spot this baseline has. A reviewer working purely off `impression_tier` would treat these the same as a stable, well-engaged excellent-tier page.

## 4. Weak picks + leakage check

**Weak picks:** any "excellent" tier page that's actually declining steeply with weak engagement is a weak pick under this baseline — it looks top-priority by traffic alone, but a smarter method (see w05) separates it into its own higher-urgency archetype instead of burying it inside a generic "excellent" bucket.

**Leakage check:** `impression_tier` is a precalculated bucket of `impressions_90d`, which is a legitimate observed signal — no product-decision flag or future-window data is involved.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words (observed / directional / decision-support), never causal or 'predicting Google'